In [ ]:
Orders

# CREATE TABLE shopey.orders (

# order_id SERIAL PRIMARY KEY,

# customer_id INT NOT NULL REFERENCES shopey.customers(customer_id),

# order_date TIMESTAMP DEFAULT NOW(),

# status VARCHAR(20) CHECK (status IN ('Pending', 'Confirmed', 'Shipped', 'Delivered', 'Cancelled')) DEFAULT 'Pending' );

# Order Lines

# CREATE TABLE shopey.order_lines (

# line_id SERIAL PRIMARY KEY,

# order_id INT NOT NULL REFERENCES shopey.orders(order_id),

# product_id INT NOT NULL REFERENCES shopey.products(product_id),

# quantity INT NOT NULL CHECK (quantity > 0),

# ); unit_price NUMERIC(10,2) NOT NULL

# Payments

# CREATE TABLE shopey.payments (

# payment_id SERIAL PRIMARY KEY,

# order_id INT UNIQUE NOT NULL REFERENCES shopey.orders(order_id),

# payment_date TIMESTAMP,

# method VARCHAR(50) CHECK (method IN ('Card', 'PayPal', 'Bank Transfer', 'Wallet')),
# amount  NUMERIC(19,2) NOT NULL,
# status  VARCHAR(20) CHECK (status IN ('Pending', 'Paid', 'Failed', 'Refunded')) DEFAULT 'Pending'





In [ ]:

#Based on the requirements and schema shown in the image, the SQL query would be:

import sqlite3
import pandas as pd


# ============================================================
# CREATE IN-MEMORY DATABASE
# ============================================================

conn = sqlite3.connect(':memory:')

cursor = conn.cursor()


# ============================================================
# CREATE TABLES
# ============================================================

cursor.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    email TEXT
)
""")

cursor.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    status TEXT
)
""")

cursor.execute("""
CREATE TABLE order_lines (
    line_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    unit_price REAL
)
""")


# ============================================================
# INSERT SAMPLE DATA
# ============================================================

customers_data = [
    (1, 'Alice', 'Smith', 'alice@email.com'),
    (2, 'Bob', 'Johnson', 'bob@email.com'),
    (3, 'Carol', 'White', 'carol@email.com')
]

orders_data = [
    (1, 1, '2025-01-01', 'Delivered'),
    (2, 1, '2025-01-02', 'Delivered'),
    (3, 2, '2025-01-03', 'Delivered'),
    (4, 2, '2025-01-04', 'Delivered'),
    (5, 3, '2025-01-05', 'Delivered')
]

order_lines_data = [
    (1, 1, 101, 2, 200.00),
    (2, 1, 102, 1, 150.50),
    (3, 2, 103, 3, 180.00),
    (4, 3, 104, 2, 220.00),
    (5, 4, 105, 1, 340.00),
    (6, 5, 106, 5, 120.00)
]

cursor.executemany(
    "INSERT INTO customers VALUES (?, ?, ?, ?)",
    customers_data
)

cursor.executemany(
    "INSERT INTO orders VALUES (?, ?, ?, ?)",
    orders_data
)

cursor.executemany(
    "INSERT INTO order_lines VALUES (?, ?, ?, ?, ?)",
    order_lines_data
)

conn.commit()


# ============================================================
# FINAL QUERY
# ============================================================

query = """

SELECT
    c.first_name || ' ' || c.last_name AS customer_name,

    COUNT(DISTINCT o.order_id) AS total_orders,

    ROUND(SUM(ol.quantity * ol.unit_price), 2) AS total_spent,

    RANK() OVER (
        ORDER BY SUM(ol.quantity * ol.unit_price) DESC
    ) AS customer_rank

FROM customers c

JOIN orders o
    ON c.customer_id = o.customer_id

JOIN order_lines ol
    ON o.order_id = ol.order_id

GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name

HAVING COUNT(DISTINCT o.order_id) >= 1

ORDER BY customer_rank ASC

"""


# ============================================================
# DISPLAY RESULTS
# ============================================================

df = pd.read_sql_query(query, conn)

print("ShopEY Customer Spending & Ranking Report\n")

display(df)

Based on the requirements and schema shown in the image, the SQL query would be:
ShopEY Customer Spending & Ranking Report



,customer_name,total_orders,total_spent,customer_rank
0,Alice Smith,2,1090.5,1
1,Bob Johnson,2,780.0,2
2,Carol White,1,600.0,3
